# EasyOCR + 辞書修正パイプライン
マルハン曲田仕掛け一覧画像をOCRし、既知語彙で精度向上させます

## 1. 必要なライブラリのインポート

In [11]:
import easyocr
import cv2
import json
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict
import pandas as pd
from IPython.display import display, Image as IPImage

print("✅ ライブラリインポート完了")

✅ ライブラリインポート完了


## 2. 修正ルール辞書の定義

In [12]:
# マルハン曲田頻出語彙の修正ルール
CORRECTION_RULES = {
    # パチンコ機種名
    'ニャンギラス': ['ニャンギラス', 'ニャンギラ', 'ニャン'],
    'マギレコ': ['マギレコ', 'マギ'],
    'モンハンライズ': ['モンハンライズ', 'モンハン'],
    'バイオ5': ['バイオ5', 'バイオ'],
    'ジャグラー': ['ジャグラー', 'ジャグ'],
    'からくり': ['からくり', 'からく'],
    'アズールレーン': ['アズールレーン', 'アズール'],
    'スタディライト': ['スタディライト', 'スタデ'],
    'シャーマシング': ['シャーマシング', 'シャー'],
    'ゴーギャッシュ': ['ゴーギャッシュ', 'ゴー'],
    '沖ドキ': ['沖ドキ', '沖'],
    'キンハナ': ['キンハナ', 'キン'],
    'エウレカ': ['エウレカ', 'エウ'],
    'カバネリ': ['カバネリ', 'カバ'],
    'デビルメイクライ': ['デビルメイクライ', 'デビル'],
    'TOL.LOVE6': ['TOL.LOVE6', 'TOL'],
    'ガリラ': ['ガリラ', 'ガリ'],
    'ディスク2': ['ディスク2', 'ディス'],
    'パンドリ': ['パンドリ', 'バンドリ'],
    'SAO': ['SAO', 'ＳＡＯ'],
    'SBJ': ['SBJ', 'ＳＢＪ'],
    
    # 曜日関連
    '本日': ['本日', '木曜日'],
    '水曜日': ['水曜日', '水'],
    '木曜日': ['木曜日', '木'],
    '金曜日': ['金曜日', '金'],
    '土曜日': ['土曜日', '土'],
    '日曜日': ['日曜日', '日'],
    '月曜日': ['月曜日', '月'],
}

print(f"✅ {len(CORRECTION_RULES)}個の修正ルールを定義")

✅ 28個の修正ルールを定義


## 3. ユーティリティ関数の定義

In [13]:
def correct_text(text: str, rules: Dict[str, List[str]]) -> str:
    """辞書ルールでテキストを修正"""
    for correct_word, error_patterns in rules.items():
        for pattern in error_patterns:
            if pattern in text:
                text = text.replace(pattern, correct_word)
    return text


def analyze_results(results: List[Tuple]) -> Dict:
    """OCR結果を分析"""
    texts = [item[1] for item in results]
    confidences = [item[2] for item in results]
    
    low_confidence = [(text, conf) for (_, text, conf) in results if conf < 0.7]
    
    analysis = {
        'total_lines': len(results),
        'avg_confidence': sum(confidences) / len(confidences) if confidences else 0,
        'min_confidence': min(confidences) if confidences else 0,
        'max_confidence': max(confidences) if confidences else 0,
        'low_confidence_count': len(low_confidence),
        'low_confidence_details': low_confidence,
        'word_frequency': Counter(texts).most_common(30),
    }
    
    return analysis


def print_analysis(analysis: Dict):
    """分析結果を表示"""
    print("\n" + "="*60)
    print("📊 OCR分析結果")
    print("="*60)
    print(f"検出行数: {analysis['total_lines']}")
    print(f"平均信頼度: {analysis['avg_confidence']:.2%}")
    print(f"信頼度範囲: {analysis['min_confidence']:.2%} - {analysis['max_confidence']:.2%}")
    print(f"信頼度70%未満: {analysis['low_confidence_count']}件\n")
    
    if analysis['low_confidence_details']:
        print("⚠️  信頼度が低いテキスト (トップ10):")
        for text, conf in sorted(analysis['low_confidence_details'], 
                                 key=lambda x: x[1])[:10]:
            print(f"  {conf:.2%}: '{text}'")
    
    print("\n📈 単語出現頻度 (トップ20):")
    for word, count in analysis['word_frequency'][:20]:
        print(f"  {count:3d}回: '{word}'")

print("✅ ユーティリティ関数定義完了")

✅ ユーティリティ関数定義完了


## 4. 画像パスの指定

In [14]:
# 📝 ここで画像ファイルのパスを指定してください
image_path = "G_MRiMTagAA2hmy.jpg"  # <- 編集: 画像のパスを入力

# パスの確認
image_file = Path(image_path)
if image_file.exists():
    print(f"✅ 画像ファイルが見つかりました: {image_file}")
    print(f"   ファイルサイズ: {image_file.stat().st_size / 1024:.1f} KB")
else:
    print(f"❌ エラー: ファイルが見つかりません: {image_path}")
    print("\n画像パスを指定し直してください")

✅ 画像ファイルが見つかりました: G_MRiMTagAA2hmy.jpg
   ファイルサイズ: 1005.2 KB


## 5. EasyOCRを実行

In [15]:
print("🔧 EasyOCRを初期化中... (GPU: True)")
print("初回実行時はモデルのダウンロードに時間がかかります")

reader = easyocr.Reader(['ja'], gpu=True)
print("✅ EasyOCR初期化完了\n")

print(f"📸 画像を処理中: {image_path}")
results = reader.readtext(image_path)
print(f"✅ OCR完了: {len(results)}行を検出")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


🔧 EasyOCRを初期化中... (GPU: True)
初回実行時はモデルのダウンロードに時間がかかります
✅ EasyOCR初期化完了

📸 画像を処理中: G_MRiMTagAA2hmy.jpg


c:\Users\apto117\Documents\pachinko-analyzer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ OCR完了: 279行を検出


## 6. 結果を分析

In [16]:
analysis = analyze_results(results)
print_analysis(analysis)


📊 OCR分析結果
検出行数: 279
平均信頼度: 83.45%
信頼度範囲: 0.00% - 100.00%
信頼度70%未満: 61件

⚠️  信頼度が低いテキスト (トップ10):
  0.00%: '一】・'
  0.03%: '一,・'
  0.11%: '下_のリヒる'
  0.52%: '下。_のリ丘る'
  1.18%: 'ヴヴヴ 下_~リヒる'
  1.33%: '全一ロ川じ'
  3.21%: '全一下_のリヒる'
  5.00%: '全一下_のリ丘る 秘宝伝'
  5.65%: '日0メ (クレア日、 エヴァ日丁など)'
  7.32%: '8'

📈 単語出現頻度 (トップ20):
    6回: 'ニャンギラス'
    6回: '月曜日一列全仕掛け'
    6回: '月曜日'
    6回: '火曜日'
    5回: '金曜日'
    5回: '全一鉄拳6'
    4回: '台)'
    4回: '木曜日一他の曜日仕掛けのいずれか'
    4回: '木曜日'
    4回: '金曜日一列イチ以上'
    3回: '土曜日'
    3回: '日曜日'
    3回: '火曜日一角0'
    3回: '7の付く日'
    3回: 'が発動一角0'
    3回: '全一新鬼武者3'
    3回: 'ウルミラ'
    2回: '月日ゾロ目'
    2回: '全一ジャグラー全機種全台'
    2回: '全一東リべ'


## 7. テキストを修正前後で比較

In [17]:
# 修正前後のテキストを比較
print("\n" + "="*60)
print("📝 修正前後の比較（信頼度70%未満のもの）")
print("="*60)

comparison_data = []
for (bbox, text, confidence) in results:
    if confidence < 0.7:
        corrected = correct_text(text, CORRECTION_RULES)
        comparison_data.append({
            '信頼度': f"{confidence:.2%}",
            '修正前': text,
            '修正後': corrected,
            '変更': '✏️' if text != corrected else ''
        })

if comparison_data:
    df = pd.DataFrame(comparison_data)
    display(df)
else:
    print("信頼度70%未満のテキストはありません")


📝 修正前後の比較（信頼度70%未満のもの）


,信頼度,修正前,修正後,変更
0,62.71%,全つかぐや様,全つかぐや様,
1,1.33%,全一ロ川じ,全一ロ川じ,
2,43.25%,3日,3日曜日,✏️
3,62.23%,2ト北斗転生側一角5,2ト北斗転生側一角5,
4,66.09%,火曜日一角0一ジャグラー角6 その,火曜日曜日一角0一ジャグラーラー角6 その,✏️
...,...,...,...,...
56,19.06%,'5日」,'5日曜日」,✏️
57,45.23%,3下東京喰種側一角呂,3下東京喰種側一角呂,
58,45.15%,全一5日」,全一5日曜日」,✏️
59,69.44%,全一東リべ,全一東リべ,


## 8. 修正後のテキストを表示

In [18]:
print("\n" + "="*60)
print("✨ 修正後のテキスト（全行）")
print("="*60 + "\n")

corrected_text_lines = []
for (bbox, text, confidence) in results:
    corrected = correct_text(text, CORRECTION_RULES)
    corrected_text_lines.append(corrected)
    print(f"[{confidence:.2%}] {corrected}")

corrected_full_text = '\n'.join(corrected_text_lines)


✨ 修正後のテキスト（全行）

[86.56%] マルハン蒲田 仕掛け一覧 (ぽこ独自調べ)
[98.45%] 1月曜日
[93.83%] メガ蒲田7
[85.09%] メガ蒲田1
[96.18%] 月曜日日曜日ゾロ目
[97.15%] 月曜日日曜日ゾロ目
[99.92%] 1日曜日
[90.07%] +264,800
[70.82%] 全一ジャグラーラー全機種全台
[96.20%] +105,100
[81.52%] 全一ジャグラーラー全機種全台
[99.73%] ニャンギラスギラスス
[99.60%] ニャンギラスギラスス
[82.10%] 2日曜日_
[93.62%] 金曜日曜日曜日曜日
[98.52%] +128,800
[95.94%] 全一マギレコレコ
[99.89%] 金曜日曜日曜日曜日
[98.55%] +20,900
[62.71%] 全つかぐや様
[1.33%] 全一ロ川じ
[98.33%] アズレン シャーマシングマン
[43.25%] 3日曜日
[99.97%] 土曜日曜日曜日曜日
[88.36%] +39,300
[72.50%] 全一モンキー
[99.96%] 土曜日曜日曜日曜日
[99.99%] プリズムナナ
[96.69%] 4日曜日
[99.94%] 日曜日曜日曜日
[88.41%] 全一防振り
[99.94%] 日曜日曜日曜日
[97.21%] +57,400
[94.18%] 全一モンハンライズライズ
[99.51%] 月曜日曜日曜日一列全仕掛け
[99.22%] 月曜日曜日曜日一列全仕掛け
[73.14%] モンキー
[98.88%] (2041-2052番台)
[94.67%] 銭形5~アズレン (2256-2268番
[97.81%] モンハンライズライズ (2089-2101番台)
[99.74%] 5日曜日
[99.95%] 月曜日曜日曜日
[94.60%] +321,500
[99.84%] 月曜日曜日曜日
[98.23%] +2,900
[99.96%] 台)
[75.06%] 新鬼武者3~戦国乙女 (2115-2128
[93.45%] 全一銭形5 秘宝伝 化物語
[99.88%] アズレ
[99.30%] 番台)
[95.88%] ン
[97.99%] 全一鉄拳6
[99.84%] モンハンライズライズ
[9

## 9. 結果をファイルに保存

In [19]:
base_name = Path(image_path).stem

# テキスト形式（修正後）
output_txt = f"{base_name}_ocr_corrected.txt"
with open(output_txt, 'w', encoding='utf-8') as f:
    f.write(corrected_full_text)
print(f"✅ {output_txt} を保存")

# テキスト形式（修正前）
output_txt_raw = f"{base_name}_ocr_raw.txt"
with open(output_txt_raw, 'w', encoding='utf-8') as f:
    f.write('\n'.join([item[1] for item in results]))
print(f"✅ {output_txt_raw} を保存")

# JSON形式（修正後）
output_json = f"{base_name}_ocr_corrected.json"
json_data = []
for (bbox, text, confidence) in results:
    corrected = correct_text(text, CORRECTION_RULES)
    json_data.append({
        'text': corrected,
        'confidence': float(confidence),
        'bbox': [[float(coord) for coord in point] for point in bbox]
    })

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)
print(f"✅ {output_json} を保存")

print("\n" + "="*60)
print("✨ 処理完了！")
print("="*60)

✅ G_MRiMTagAA2hmy_ocr_corrected.txt を保存
✅ G_MRiMTagAA2hmy_ocr_raw.txt を保存
✅ G_MRiMTagAA2hmy_ocr_corrected.json を保存

✨ 処理完了！


## 10. 修正ルールのカスタマイズ（オプション）

新しい誤認識パターンが見つかった場合、以下のセルで追加修正ルールを定義できます

In [20]:
# 📝 ここで追加の修正ルールを定義
ADDITIONAL_RULES = {
    # 例: '正しい言葉': ['誤認識パターン1', '誤認識パターン2']
}

# 既存のルールにマージ
CORRECTION_RULES.update(ADDITIONAL_RULES)
print(f"✅ {len(ADDITIONAL_RULES)}個の追加ルールをマージ")

# 修正を再実行
if ADDITIONAL_RULES:
    corrected_text_lines = []
    for (bbox, text, confidence) in results:
        corrected = correct_text(text, CORRECTION_RULES)
        corrected_text_lines.append(corrected)
    corrected_full_text = '\n'.join(corrected_text_lines)
    print("✅ テキストを再修正しました")

✅ 0個の追加ルールをマージ
